In [55]:
import pandas as pd
import numpy as np
import kagglehub
import re
from astroquery.gaia import Gaia
# french mirror of thre ea gaia data sets
from astroquery.vizier import Vizier
import requests
from bs4 import BeautifulSoup
import json
from datetime import datetime
import os
import matplotlib.pyplot as plt
import seaborn as sns

accounts

ESA > https://psaftp.esac.esa.int/WebInterface/login.html
username = ekelly
pwd = SlieveMor01!


GAIA - catalog sizes  > https://cdn.gea.esac.esa.int/Gaia/gdr3/_catalogue_sizes.txt

In [56]:
#defining global variable path to files and data, it is cvalled thropughout modify this path to match your local path
path = './'

## Project Objective

Project Question = *Does the Sun have a twin or a closely related cousin?*

Yhe goal of this project is to see can we determine if there another star in the universe that is identical or similar to our Sun, that we know of and have measurements, metrics for.

We will tak astronmical data from avariety of diffeent public sources. 

1) Stellar Hosts - nasa - one row per star.
2) pscompars (planetary system composite parameters)= nasa
4) GAIA - DR3 - for around 1.46 billion (1.46 109) sources >> https://www.cosmos.esa.int/web/gaia/dr3 
    Sources in the Gaia Catalogue are all identified through the Gaia Source Identifier, i.e., the source_id field in the various tables in the Gaia Archive > is from Dr3 is different to DR2 etc
    3a) 3) I/355/paramsp (astrphysical_parameters) - ESA (GAIA) - (1.5 million rows)
    https://www.aanda.org/articles/aa/full_html/2023/06/aa43688-22/aa43688-22.html 
    - this is a subset of calculated data from the DR3 dataset it is a table within the DR3 dataset
    - - this is what we will use
4) LAMOST (Large Sky Area Multi object Fiber Spectroscope) - China - https://www.lamost.org/lmusers/user/
    latest data DR13 requires approval and login
    DR10 is downloadable - so using that

had problems with gaia IDs and bigint data types, gett exponential numbers,  conversion issues, losing digits. decided to use strings for those 19 digit ID numbers. theyh are also identifies, labels, so not used mathematically, so no calculations use dno them as such string is fine.

FI Files  - come as indiviaul filess, have to be merged standane sets fo data.

#### 1) Dataset - NASA >  Stellar Hosts

Data set taken from https://exoplanetarchive.ipac.caltech.edu/ 

We will down lai the 'Stellar Hosts' dataset. This dataset is 1 row per star. It is determined by the search for exoplanets, sok each star will have 0 to > 0 number of planets orbitting it.

In [ ]:
# after a bit of playing around tihe the url format we get the url for csv format for the entire stellar hosts dataset
# we download in csv format
# This uses their preferred TAP API, like a sql command to directly query the database, and we get it in a clean csv.
url = "https://exoplanetarchive.ipac.caltech.edu/TAP/sync?query=select+*+from+stellarhosts&format=csv"
stellar_host_df = pd.read_csv(url)

stellar_host_df.shape

# create a lcoal copy in csv as had issue with server going offline and access data. so save lcoal copy in case needed
# wil put a time stampe on the filename for uniqueness
datetime_stamp = datetime.now().strftime("%d%m%y")
sh_csv_file_name = f"{path}stellarhosts_{datetime_stamp}.csv"
stellar_host_df.to_csv(sh_csv_file_name, index=False)
print(f"saved {sh_csv_file_name}")

# NOTE th is nto one row per star >> all sceintific measurements are kept and sources are different, so needs to be filered....
# may add update a paramater, i.e. update mass of a star , added in a new row, old row kept...

saved stellarhosts_091225.csv


In [ ]:
# check downloadded fine and get feel for shape of data downloaded
stellar_host_df.head()

stellar_host_df.shape

(46870, 136)

Need to get the uniqwue values for GAIA_IDs. We will sue those to download other datasets. Other datasets are to big, in TBs in siuze, and this will allow us filter just what we need, reduce downlaod time and make ti more manageable

In [58]:
# get colimns containign the string 'gaia' from the dataframe
# gaia IDs are the column we will merge against and also pivit around across daatsets. This is a unique identifier used by the ESA
# we willa also use gaia ID to filter data we down load from other sources, to limit file size to only relevant data/rows.
gaia_cols = [col for col in stellar_host_df.columns if'gaia' in col.lower()]

print(f"gaia columns  : {gaia_cols}")

gaia columns  : ['sy_gaiamag', 'sy_gaiamagerr1', 'sy_gaiamagerr2', 'gaia_dr2_id', 'gaia_dr3_id']


Check the data format. we will leave as astr object types as convertign to int rounds and we havhe huge 19 digit numbers, we can do string matches across datasets

As per gudielines on EAA site there is no direct 

Sources in the Gaia Catalogue are all identified through the Gaia Source Identifier, i.e., the source_id field in the various tables in the Gaia Archive. the source list for Gaia DR3 should be treated as independent from Gaia DR2 and from Gaia DR1. With each new Gaia data release, the source list is becoming progressively more stable

Refer to [Gaia Data Release 3 (Gaia DR3)](https://www.cosmos.esa.int/web/gaia/dr3) for details from above amnd futher details re the data set.

In [ ]:
 head(10)

0    Gaia DR3 2128190453050802048
1    Gaia DR3 2105930840143687680
2    Gaia DR3 2077595394707557120
3    Gaia DR3 2085724496490595584
4    Gaia DR3 2129158435598210816
5    Gaia DR3 2086414268238649984
6    Gaia DR3 2052582432887909376
7    Gaia DR3 2052942900903300352
8    Gaia DR3 2130211080541825024
9    Gaia DR3 2128079230577866880
Name: gaia_dr3_id, dtype: object

strip the text and number not needed, so onyl get 19n digit number at the end of string, use regex expressino to filter the cell values

want to get the GAIA ID numebrs, minus the strings

Problem
need to match onyl yrh gaia IDs 8numbers) so can downlaod onyl the data we wnat from gaia downlaod
DR3 and DR4 are nto 100% identical m and this is as expected as ID get updated as data is updated sources are split, uydpated data changes thigns etc so it loks like our data is intact and we have ths striped gaiaa ID

In [61]:
stellar_host_df_2 = stellar_host_df.copy()

#problemnumbertyep19digitnumbercompareasastring?
# need to use numpyarraytosuenjkmber
# pandas os displying 19 digit numbers a
# print(stellar_host_df_2['gaia_dr2_id'].head(10).tolist())
# print(stellar_host_df_2['gaia_dr3_id'].head(10).tolist())

# http://regex101.com
# $ start at the end of thestring, only match the last part,
# smatch 10 to 20 digits in a row, ignore the space, so we don't get the 2 from DR2, so contiguous, between 10 and 20 characters and at the end of the string 
# stop at space
stellar_host_df_2['dr2_num'] = stellar_host_df_2['gaia_dr2_id'].astype(str).str.extract(r'(\d{10,20})', expand=False)
stellar_host_df_2['dr3_num'] = stellar_host_df_2['gaia_dr3_id'].astype(str).str.extract(r'(\d{10,20})', expand=False)

print(stellar_host_df_2['dr2_num'].head(10))
print(stellar_host_df_2['dr3_num'].head(10))

# stellar_host_df_2['dr2_equals_dr3'] = stellar_host_df_2['dr2_num'] == stellar_host_df_2['dr3_num']

# stellar_host_df_2 = stellar_host_df_2.dropna(subset=['dr3_num'])stellar_host_df_2['dr2_equals_dr3'].value_counts()

0    2128190453050802048
1    2105930840143687680
2    2077595394707557120
3    2085724496490595584
4    2129158435598210816
5    2086414268238649984
6    2052582432887909376
7    2052942900903300352
8    2130211080541825024
9    2128079230577866880
Name: dr2_num, dtype: object
0    2128190453050802048
1    2105930840143687680
2    2077595394707557120
3    2085724496490595584
4    2129158435598210816
5    2086414268238649984
6    2052582432887909376
7    2052942900903300352
8    2130211080541825024
9    2128079230577866880
Name: dr3_num, dtype: object


Check to see fi cell values are as we expect, identify ant problematoc cell data

In [62]:
# check to see if gaia Ids are as we expect and what ros contain data we are not expecting and are problematic
# Convert to string for consistent checks
col = stellar_host_df_2['dr3_num'].astype(str)

diagnostics = {
    "total_rows": len(col),

    # Missing or NA-like
    "is_na": col.isna().sum(),
    "empty_string": (col.str.strip() == "").sum(),
    "literal_nan_or_none": col.str.lower().isin(["nan", "none"]).sum(),

    # Invalid characters (non-digits)
    "contains_nondigits": col.str.contains(r"\D", regex=True).sum(),

    # Has a valid digit sequence (extractable)
    "contains_digit_block_10_20": col.str.contains(r"\d{10,20}", regex=True).sum(),

    # Fullmatch valid Gaia ID (exact digits only)
    "valid_fullmatch": col.str.fullmatch(r"\d{10,20}").sum(),

    # Wrong length but digits-only
    "digits_only_wrong_length": (
        col.str.fullmatch(r"\d+").fillna(False) &
        ~col.str.fullmatch(r"\d{10,20}")
    ).sum(),
}

pd.Series(diagnostics)


total_rows                    46870
is_na                             0
empty_string                      0
literal_nan_or_none            1738
contains_nondigits             1738
contains_digit_block_10_20    45132
valid_fullmatch               45132
digits_only_wrong_length          0
dtype: int64

In [73]:
mask_literal = stellar_host_df_2['dr3_num'].astype(str).str.lower().isin(["nan", "none"])
problem_literal = stellar_host_df_2[mask_literal]

problem_literal


,hostname,hd_name,hip_name,tic_id,st_refname,sy_refname,ra,rastr,dec,decstr,...,sy_kepmagerr2,st_rotp,st_rotperr1,st_rotperr2,st_rotplim,gaia_dr2_id,gaia_dr3_id,cb_flag,dr2_num,dr3_num
2362,KMT-2023-BLG-0548L,NaN,NaN,NaN,<a refstr=HAN_ET_AL__2025 href=https://ui.adsa...,<a refstr=HAN_ET_AL__2025 href=https://ui.adsa...,270.347625,18h01m23.43s,-27.106750,-27d06m24.30s,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN
2363,KMT-2023-BLG-0548L,NaN,NaN,NaN,<a refstr=HAN_ET_AL__2025 href=https://ui.adsa...,<a refstr=HAN_ET_AL__2025 href=https://ui.adsa...,270.347625,18h01m23.43s,-27.106750,-27d06m24.30s,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN
2825,KMT-2023-BLG-1896L,NaN,NaN,NaN,<a refstr=HAN_ET_AL__2025 href=https://ui.adsa...,<a refstr=HAN_ET_AL__2025 href=https://ui.adsa...,271.031750,18h04m07.62s,-26.958919,-26d57m32.11s,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN
2845,KMT-2023-BLG-1896L,NaN,NaN,NaN,<a refstr=HAN_ET_AL__2025 href=https://ui.adsa...,<a refstr=HAN_ET_AL__2025 href=https://ui.adsa...,271.031750,18h04m07.62s,-26.958919,-26d57m32.11s,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN
2854,KMT-2020-BLG-0414L,NaN,NaN,NaN,<a refstr=ZHANG_ET_AL__2024 href=https://ui.ad...,<a refstr=ZANG_ET_AL__2021 href=https://ui.ads...,271.915000,18h07m39.60s,-28.485222,-28d29m06.8s,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46865,KMT-2018-BLG-0748L,NaN,NaN,NaN,<a refstr=HAN_ET_AL__2020 href=https://ui.adsa...,<a refstr=HAN_ET_AL__2020 href=https://ui.adsa...,267.874500,17h51m29.88s,-30.646389,-30d38m47.00s,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN
46866,KMT-2019-BLG-0842L,NaN,NaN,NaN,<a refstr=JUNG_ET_AL__2020 href=https://ui.ads...,<a refstr=JUNG_ET_AL__2020 href=https://ui.ads...,268.458458,17h53m50.03s,-29.877439,-29d52m38.78s,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN
46867,KMT-2016-BLG-2364L,NaN,NaN,NaN,<a refstr=HAN_ET_AL__2020 href=https://ui.adsa...,<a refstr=HAN_ET_AL__2020 href=https://ui.adsa...,265.715667,17h42m51.76s,-27.435561,-27d26m08.02s,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN
46868,KMT-2016-BLG-2397L,NaN,NaN,NaN,<a refstr=HAN_ET_AL__2020 href=https://ui.adsa...,<a refstr=HAN_ET_AL__2020 href=https://ui.adsa...,266.212542,17h44m51.01s,-23.201381,-23d12m04.97s,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN


Remove problematic cells, and ensure shape # of rows matches shape value for rows continaing correct data structure as per earlier diagnostic

In [20]:
# remove the probllematic gaia ID rows

sh_clean_gaia_df = stellar_host_df_2[stellar_host_df_2['dr3_num'].astype(str).str.fullmatch(r'\d{10,20}')].copy()

#checl rows have been removed
sh_clean_gaia_df.shape

# sh_clean_gaia_df['dr3_num'].head()


(45132, 138)

Create a list containing the gaia Ids, we will sue thsi them to filter future dataset downlaods , so we onyl get rows and data that is usefule and relevant to us

In [39]:

# put gaia ID values into a list so we cna filter future dataset downloads and reduce downlaod size 
# gaia_id_dr3_list = sh_clean_gaia_df['dr3_num'].tolist()

print(f"length of gaia Id list - {len(gaia_id_dr3_list)}")
gaia_id_dr3_list[:5]


length of gaia Id list - 45132


['2128190453050802048',
 '2105930840143687680',
 '2077595394707557120',
 '2085724496490595584',
 '2129158435598210816']

### 2) PSCompars

In [64]:
# downlaod using ADQL and the tap server
# we'll save as csv aswell as have had problems with downtime and mainenance on  servers, so as a backup
# this is the full dataset
url = "https://exoplanetarchive.ipac.caltech.edu/TAP/sync?query=select+*+from+pscomppars&format=csv"
pscompars_df = pd.read_csv(url)

# use dat time stanpe to give us unique id for the csv
datetime_stamp = datetime.now().strftime("%d%m%y_%H")
pscompars_csv_file_name = f"{path}pscompars_{datetime_stamp}.csv"
pscompars_df.to_csv(pscompars_csv_file_name, index=False)
print(f"saved {pscompars_csv_file_name}")


print(pscompars_df.shape)
#pscompars_df.cols

saved ./pscompars_101225_19.csv
(6053, 683)


In [65]:
gaia_cols_pscompars = [col for col in stellar_host_df.columns if'gaia' in col.lower()]

print(f"gaia columns  : {gaia_cols_pscompars}")

gaia columns  : ['sy_gaiamag', 'sy_gaiamagerr1', 'sy_gaiamagerr2', 'gaia_dr2_id', 'gaia_dr3_id']


filter directly on stellar_hosts_df[gaia_dr3] 

code is creating anew df which contains onyl rows wher gaia_dr3_id values in pscompars is = to gaia_dr3_ids in the stellar_hosts_df, is this correct?

In [71]:
# pscompars_df['gaia_dr3_id'].head(10)
# pscompars_df['gaia_dr3_id'].dtype

print(type(pscompars_df['gaia_dr3_id'].iloc[0]))
pscompars_df['gaia_dr3_id'].apply(type).value_counts()



<class 'str'>


gaia_dr3_id
<class 'str'>      5697
<class 'float'>     356
Name: count, dtype: int64

In [51]:
# maually downlaoded paramater from gaia Vizier
filename = "gaia_pa_vizier_45k_manual_dwnld.csv"

full_path = f"{path}{filename}"
pa_gaia_df = pd.read_csv(full_path, dtype={"Source":"str"})

pa_gaia_df.shape
pa_gaia_df.head()



,Source,RA_ICRS,DE_ICRS,PQSO,PGal,Pstar,PWD,Pbin,Teff,logg,...,Rad-Flame,Lum-Flame,Mass-Flame,Age-Flame,z-Flame,Teff-HS,logg1-MSC,logg-HS,logg-S,logg2-MSC
0,4295806720,44.996155,0.005615,0.0,0.0,0.999988,0.0,0.000012,5052.9760,4.7793,...,0.5188,0.157970,NaN,NaN,0.731631,NaN,4.5381,NaN,NaN,4.5286
1,34361129088,45.004320,0.021048,0.0,0.0,0.999389,0.0,0.000611,3478.5408,4.7000,...,0.3923,0.020405,NaN,NaN,0.459134,NaN,4.6358,NaN,NaN,4.7863
2,38655544960,45.004978,0.019880,0.0,0.0,0.999806,0.0,0.000194,4708.7944,4.5588,...,0.7114,0.226447,0.77,3.273,0.635463,NaN,4.7001,NaN,NaN,4.9004
3,309238066432,44.995037,0.038152,0.0,0.0,0.999980,0.0,0.000020,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,343597448960,44.963896,0.043595,0.0,0.0,0.999991,0.0,0.000009,4536.6640,4.6276,...,0.5979,0.135557,NaN,NaN,0.641090,NaN,4.5318,NaN,NaN,4.6851


In [54]:
from astroquery.gaia import Gaia
import pandas as pd
import requests
import time

# ----------------------------------------
# CHECK IF GAIA TAP IS ONLINE
# ----------------------------------------
def check_gaia_tap():
    url = "https://gea.esac.esa.int/tap-server/tap/availability"
    try:
        r = requests.get(url, timeout=5)
        if r.status_code == 200 and "available" in r.text.lower():
            print("Gaia TAP is ONLINE ✔")
            return True
        else:
            print("Gaia TAP reachable but not fully available")
            return False
    except Exception as e:
        print("Gaia TAP OFFLINE ✖ :", e)
        return False


# ----------------------------------------
# FETCH ASTROPHYSICAL PARAMETERS FOR YOUR IDs
# ----------------------------------------
def fetch_gaia_dr3_ap(gaia_ids, chunk_size=1500):
    gaia_ids = [str(x) for x in gaia_ids]  # ensure string source_ids

    n = len(gaia_ids)
    chunks = [gaia_ids[i:i + chunk_size] for i in range(0, n, chunk_size)]
    print(f"Fetching AP data in {len(chunks)} chunks...")

    dfs = []

    for idx, chunk in enumerate(chunks, 1):
        print(f"\nChunk {idx}/{len(chunks)}...")

        id_str = ",".join(chunk)

        query = f"""
        SELECT *
        FROM gaiadr3.astrophysical_parameters
        WHERE source_id IN ({id_str})
        """

        job = Gaia.launch_job_async(query)
        result = job.get_results()

        df = result.to_pandas()
        dfs.append(df)

        time.sleep(0.25)   # polite delay to avoid rate limiting

    # Combine all returned rows
    final_df = pd.concat(dfs, ignore_index=True)
    return final_df


# ----------------------------------------
# MAIN EXECUTION
# ----------------------------------------
if check_gaia_tap():
    print("Downloading Gaia DR3 AP rows for your stars...")

    gaia_ap_subset = fetch_gaia_dr3_ap(gaia_id_dr3_list)

    print("\nDONE ✔")
    print("Final dataframe shape:", gaia_ap_subset.shape)

    gaia_ap_subset.to_csv("gaia_dr3_astrophysical_parameters_subset.csv", index=False)
    print("Saved → gaia_dr3_astrophysical_parameters_subset.csv")

else:
    print("TAP offline — aborting")


Gaia TAP is ONLINE ✔
Fetching AP data in 31 chunks...

Chunk 1/31...


INFO: Query finished. [astroquery.utils.tap.core]

Chunk 2/31...


INFO: Query finished. [astroquery.utils.tap.core]

Chunk 3/31...


INFO: Query finished. [astroquery.utils.tap.core]

Chunk 4/31...


INFO: Query finished. [astroquery.utils.tap.core]

Chunk 5/31...


...

INFO: Query finished. [astroquery.utils.tap.core]

Chunk 28/31...


INFO: Query finished. [astroquery.utils.tap.core]

Chunk 29/31...


INFO: Query finished. [astroquery.utils.tap.core]

Chunk 30/31...


INFO: Query finished. [astroquery.utils.tap.core]

Chunk 31/31...


INFO: Query finished. [astroquery.utils.tap.core]

DONE ✔
Final dataframe shape: (34881, 226)
Saved → gaia_dr3_astrophysical_parameters_subset.csv
